# 2.3 怎麼混合上下文特徵？


[2.2](https://birdhackor.github.io/tiny-perceptron-vlm/2.2.html) 把前文的數字按位置排好，但還沒讓它們互相影響。現在像調飲料配方一樣，每種材料給一個權重，乘完再相加；一種配方產生一個輸出特徵，多種配方就產生一排新特徵。這樣模型纔有機會把「前面出現我、最後出現喝」兩條線索放在同一個計算裡，而不是隻保存兩張卡。

本節先縮小成三個輸入特徵 `[1,2,4]`，做兩套配方。第一套權重 `[1,0,2]`、偏移1，結果為 `1×1+2×0+4×2+1=10`；第二套 `[0,-1,1]`、偏移0，結果為 `0-2+4=2`。偏移量（bias）是一個不看輸入也會加上的常數。加權和與矩陣背景見 [W.5](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.5)。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

layer = nn.Linear(3, 2)
with torch.no_grad():
    layer.weight.copy_(torch.tensor([[1.0, 0.0, 2.0], [0.0, -1.0, 1.0]]))
    layer.bias.copy_(torch.tensor([1.0, 0.0]))
x = torch.tensor([[1.0, 2.0, 4.0]])
y = layer(x)
manual = x @ layer.weight.T + layer.bias
print(layer.weight.shape, y)
assert torch.allclose(y, manual)

輸入x是一筆三特徵，形狀 `[1,3]`。`nn.Linear(3,2)` 建立三個輸入、兩個輸出的線性層，保存兩套可調權重。我們手動設定數字，輸出便可精確核對為 `[[10,2]]`。`manual` 是同一件事的展開：`@` 將對應特徵相乘再加總，`.T` 把權重表的列欄交換，最後每筆加同一套偏移。

為什麼需要轉置？PyTorch保存的權重是 `[out,in]`，也就是 `[2,3]`：每列一套輸出配方，每列包含三個輸入權重。左邊x是 `[1,3]`，相乘的內側格數必須相等，所以右邊要轉成 `[3,2]`，結果纔是 `[1,2]`。這個非方陣例子能讓轉置方向看清楚；若都用3×3，有時寫反也不會形狀報錯，卻算出另一套配方。

Linear沿輸入的最後一個特徵軸工作，前面的筆數和位置分組保留。例如 `[2,4,3]` 交給這層會變 `[2,4,2]`，每個位置都共用同一套三到二的配方。如果想混合 [2.2](https://birdhackor.github.io/tiny-perceptron-vlm/2.2.html) 的整段前文，則先拼成長卡，再讓Linear的輸入數等於C×D。這裡「混合」不是把所有筆資料混在一起，而是每筆各自使用同樣的規則。

目前例子的數字由人指定，後續訓練會依答案代價調整 `weight` 與 `bias`，求導與更新的前置見 [1.11](https://birdhackor.github.io/tiny-perceptron-vlm/1.11.html) 和 [1.12](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html)。多一層只表示多一套可調計算，效果仍要用沒拿來訓練的資料驗證。

練習只把第一套配方的第三個權重從2改成3，先預測第一個輸出增加4而第二個不變，再執行核對應為 `[[14,2]]`。這次用具體數值追蹤「哪個參數影響哪個輸出」，比只看新的shape更能理解計算。
